# HW 3: Homogeneous Transformations
* Continue using the cumulative `byu_robomanip.transforms` module from HW02.
* Complete the HW03-labeled `se3` and `inv` methods.
* Run the provided cells and check each transform direction carefully.

In [1]:
from byu_robomanip import transforms as tr
import numpy as np
import sympy as sp
from byu_robomanip.visualization import VizScene
np.set_printoptions(precision=3)


# Problem # 1
### Homogeneous matrix made of only rotation about z-axis by amount $\frac{\pi}{4}$

Rotation by amount theta around the z-axis should give the following:
$$
\left[\begin{matrix}0.707 & -0.707 & 0 & 0\\ 0.707 & 0.707 & 0 & 0\\0 & 0 & 1 & 0\\0 & 0 & 0 & 1\end{matrix}\right]
$$

In [ ]:
T = tr.se3(tr.rotz(np.pi/4))
print(T)

### Translation only along the x-axis by an amount $0.5$ should give the following:
$$
\left[\begin{matrix}1 & 0 & 0 & 0.5 \\0 & 1 & 0 & 0\\0 & 0 & 1 & 0\\0 & 0 & 0 & 1\end{matrix}\right]
$$

In [ ]:
T = tr.se3(p=[0.5, 0, 0])
print(T)

### Checking the "inv" function

If we calculate a homogeneous transform with a rotation of 45 degrees about the x-axis and a general translation ($[0.5, 0.25, 0.3]^T$) we get the following:
$$
\left[\begin{matrix}1 & 0 & 0 & 0.5\\0 & 0.707106781186548 & -0.707106781186547 & 0.25\\0 & 0.707106781186547 & 0.707106781186548 & 0.3\\0 & 0 & 0 & 1\end{matrix}\right]
$$

Then, we can multiply the original transform by its inverse to check out inverse function, we should get the following:
$$
\left[\begin{matrix}1 & 0 & 0 & 0\\0 & 1.0 & 0 & 0\\0 & 0 & 1.0 & 0\\0 & 0 & 0 & 1\end{matrix}\right]
$$

In [ ]:
T = tr.se3(tr.rotx(np.pi/4), [0.5, 0.25, 0.3])
print(T)

# now we can check if we implemented "inv" correctly:
check = tr.inv(T) @ T
print("\n\n inv(T) @ T should give identity matrix:")
check[np.abs(check) < .00001] = 0 # get rid of small #s
print(check)

In [ ]:
#########  Practice

# R= tr.rotz(np.pi/2)
# P= np.matrix([5, 7, 8])
# T = np.zeros((4,4))
# T[0:3, 0:3] = R
# T[0:3, -1] = P
# T[-1, -1] = 44

# eps = 1e-10
# T[np.abs(T) < eps] = 0.0
# T1 = T

# print(T)
# R = T[0:3, 0:3]
# P = (T[0:3, -1])
# T[0:3, 0:3] = R.T
# T[0:3, -1] = -R.T@P
# T2 = T

# print(T)
# print(T1@T2)


# Transform a point in one frame to another frame

### Using DH parameters to find A matrices (for each joint):

Remember that if we combine a rotation in z, translation in z, then translation in x, and rotation in x, we should get the same result as the book for following the DH convention to move from one frame (or joint) to another as follows: 

$$
A = \left[\begin{matrix}\cos{\left(\theta \right)} & - \sin{\left(\theta \right)} \cos{\left(\alpha \right)} & \sin{\left(\alpha \right)} \sin{\left(\theta \right)} & a \cos{\left(\theta \right)}\\\sin{\left(\theta \right)} & \cos{\left(\alpha \right)} \cos{\left(\theta \right)} & - \sin{\left(\alpha \right)} \cos{\left(\theta \right)} & a \sin{\left(\theta \right)}\\0 & \sin{\left(\alpha \right)} & \cos{\left(\alpha \right)} & d\\0 & 0 & 0 & 1\end{matrix}\right]
$$

In future homework, we'll implement representations of robot arms that include this transformation. But for the test values of DH parameters below, we can perform this sequence of operations manually.  

Assuming the following DH parameters for two joints:

$\theta_1 = \frac{\pi}{8}$, $d_1 = 0$, $a_1 = 0.3$, $\alpha_1 = \frac{\pi}{2}$

$\theta_2 = \frac{\pi}{4}$, $d_2 = 0$, $a_2 = 0.3$, $\alpha_2 = 0$


The resulting homogeneous transform describing the tip (or frame 2) relate to frame 0 would be:

$$
\left[\begin{matrix}0.653281482438188 & -0.653281482438188 & 0.38268343236509 & 0.473148304484842\\0.270598050073099 & -0.270598050073099 & -0.923879532511287 & 0.195984444731456\\0.707106781186548 & 0.707106781186548 & 0 & 0.212132034355964\\0 & 0 & 0 & 1.0\end{matrix}\right]
$$

In [ ]:
# start by substituting the actual values for R and p and making a new SE3 object
# that describes the transformation from frame 0 to frame 1
import sympy as sp
theta, d, a, alpha = sp.symbols('theta, d, a, alpha')
A = sp.Matrix([
    [sp.cos(theta), -sp.sin(theta)*sp.cos(alpha),  sp.sin(alpha)*sp.sin(theta), a*sp.cos(theta)],
    [sp.sin(theta),  sp.cos(alpha)*sp.cos(theta), -sp.sin(alpha)*sp.cos(theta), a*sp.sin(theta)],
    [0,              sp.sin(alpha),                sp.cos(alpha),               d],
    [0,              0,                             0,                           1]
])

t01_subs = {theta:np.pi/8, d:0, a:0.3, alpha:np.pi/2}
t02_subs = {theta:np.pi/4, d:0, a:0.3, alpha:0}


# TODO - fill this out
# find the transformation from frame 0 to 1
if "T1_in_0" not in globals():
    T1_in_0 = A.subs(t01_subs)
# do the same thing for frame 1 to frame 2
if "T2_in_1" not in globals():
    T2_in_1 = A.subs(t02_subs)
if T1_in_0 is None or T2_in_1 is None:
    raise NotImplementedError("Complete the HW03 DH transforms")

# now we can combine the two to get a transformation that describes frame 2
# relative to frame 0
T2_in_0 = T1_in_0 @ T2_in_1

# printing the result
sp.pprint(T2_in_0)

# use the "add_frame" function to plot both frames (for joint 1 and joint 2) relative
# to a base or ground frame.

# TODO - put your visualization code here.

viz = VizScene()
frame_0 = np.eye(4)
viz.add_frame(frame_0, label = 'base', axes_label= '0')
viz.add_frame(np.array(T1_in_0, dtype=float), label= 'joint_1', axes_label= '1')
viz.add_frame(np.array(T2_in_0, dtype=float), label= 'joint_2', axes_label= '2')
viz.hold()


In [ ]:
viz = VizScene()
frame_0 = np.eye(4)
viz.add_frame(frame_0, label = 'base', axes_label= '0')
T1_in_0 = tr.se3(np.eye(3), np.array([0, 1, 1]))
T2_in_0 = tr.se3(np.eye(3), np.array([-1/2, 3/2, 1.1]))
T3_in_0 = tr.se3(tr.rotz(-np.pi/2)@tr.roty(np.pi), np.array([-1/2, 3/2, 3]))
viz.add_frame(T1_in_0, label= 'joint_1', axes_label= '1')
viz.add_frame(T2_in_0, label= 'joint_2', axes_label= '2')
viz.add_frame(T3_in_0, label= 'joint_3', axes_label= '2')


viz.hold()


In [ ]:
############ hw3 part d

viz = VizScene()
frame_0 = tr.se3(tr.roty(np.pi/2) @ tr.rotz(np.pi), np.zeros(3))
viz.add_frame(frame_0, label = 'base', axes_label= '0')

q1 = 0
q2 = 0
a1 = 1
a2 = 2.5
dh_table = [
    {"theta": q1, "d": 0, "a": a1, "alpha": np.pi/2},
    {"theta": 0, "d": q2+a2, "a": 0, "alpha": 0}]

T1_in_0 = frame_0 @ tr.dh_transform(**dh_table[0])
T2_in_0 = T1_in_0 @ tr.dh_transform(**dh_table[1])
viz.add_frame(T1_in_0, label= 'joint_1_config_0', axes_label='1')
viz.add_frame(T2_in_0, label= 'joint_2_config_0', axes_label='2')


#config 2
q1 = np.pi/3
q2 = -1
a1 = 1
a2 = 2.5
dh_table = [
    {"theta": q1, "d": 0, "a": a1, "alpha": np.pi/2},
    {"theta": 0, "d": q2+a2, "a": 0, "alpha": 0}]

T1_in_0_config_1 = frame_0 @ tr.dh_transform(**dh_table[0])
T2_in_0_config_1 = T1_in_0_config_1 @ tr.dh_transform(**dh_table[1])
viz.add_frame(T1_in_0_config_1, label= 'joint_1_config_1', axes_label='1')
viz.add_frame(T2_in_0_config_1, label= 'joint_2_config_1', axes_label='2')

animate = 0
if (animate):
    import time
    t_run = 60
    t = 0
    start = time.time()
    while (t < t_run):
        t = time.time() - start
        q1 = (1/2)*(1 + np.sin(t))*np.pi/3
        q2 = (1/2)*(1 + np.sin(t))*-1
        a1 = 1
        a2 = 2.5
        dh_table = [
            {"theta": q1, "d": 0, "a": a1, "alpha": np.pi/2},
            {"theta": 0, "d": q2+a2, "a": 0, "alpha": 0}]
        T1_in_0_config_1 = frame_0 @ tr.dh_transform(**dh_table[0])
        T2_in_0_config_1 = T1_in_0_config_1 @ tr.dh_transform(**dh_table[1])
        viz.update(As = [frame_0, T1_in_0, T2_in_0, T1_in_0_config_1, T2_in_0_config_1])

        viz.hold(1/60)
        # viz.close_viz()


viz.hold()
# viz.close_viz()


In [10]:
###### hw 3 part e
q1, q2, q3, d1, d2, d3, a1, a2, a3, alpha1, alpha2, alpha3 = sp.symbols("q1, q2, q3, d1, d2, d3, a1, a2, a3, alpha1, alpha2, alpha3")
dh_table = [
    {"theta": q1, "d": d1, "a": 0, "alpha": np.pi/2},
    {"theta": q2, "d": 0, "a": a2, "alpha": 0},
    {"theta": q3, "d": 0, "a": a3, "alpha": 0}]

A_matrix = tr.dh_transform(**dh_table[0], symbolic=True)
print('A1 = \n')
sp.pprint(A_matrix)

A_matrix = tr.dh_transform(**dh_table[1], symbolic=True)
print('\nA2 = \n')
sp.pprint(A_matrix)

A_matrix = tr.dh_transform(**dh_table[2], symbolic=True)
print('\nA3 = \n')
sp.pprint(A_matrix)

A1 = 

⎡cos(q₁)  -6.12323399573677e-17⋅sin(q₁)      1.0⋅sin(q₁)       0 ⎤
⎢                                                                ⎥
⎢sin(q₁)  6.12323399573677e-17⋅cos(q₁)       -1.0⋅cos(q₁)      0 ⎥
⎢                                                                ⎥
⎢   0                  1.0               6.12323399573677e-17  d₁⎥
⎢                                                                ⎥
⎣   0                   0                         0            1 ⎦

A2 = 

⎡cos(q₂)  -sin(q₂)  0  a₂⋅cos(q₂)⎤
⎢                                ⎥
⎢sin(q₂)  cos(q₂)   0  a₂⋅sin(q₂)⎥
⎢                                ⎥
⎢   0        0      1      0     ⎥
⎢                                ⎥
⎣   0        0      0      1     ⎦

A3 = 

⎡cos(q₃)  -sin(q₃)  0  a₃⋅cos(q₃)⎤
⎢                                ⎥
⎢sin(q₃)  cos(q₃)   0  a₃⋅sin(q₃)⎥
⎢                                ⎥
⎢   0        0      1      0     ⎥
⎢                                ⎥
⎣   0        0      0      1     ⎦


In [ ]:
######## hw part f
q1, q2, q3, d1, d2, d3, a1, a2, a3, alpha1, alpha2, alpha3 = sp.symbols("q1, q2, q3, d1, d2, d3, a1, a2, a3, alpha1, alpha2, alpha3")
dh_table = [
    {"theta": q1, "d": d1, "a": 0, "alpha": np.pi/2},
    {"theta": q2, "d": 0, "a": a2, "alpha": 0},
    {"theta": q3, "d": 0, "a": a3, "alpha": 0}]

A_matrix = tr.dh_transform(**dh_table[0], symbolic=True)
print('A1 = \n')
sp.pprint(A_matrix)

A_matrix = tr.dh_transform(**dh_table[1], symbolic=True)
print('\nA2 = \n')
sp.pprint(A_matrix)

A_matrix = tr.dh_transform(**dh_table[2], symbolic=True)
print('\nA3 = \n')
sp.pprint(A_matrix)